# Spark basics

Calculate order totals with DataFrames and Spark SQL.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("spark-basics-notebook")
         .master("local[1]")
         .config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", "1")
         .getOrCreate())

orders = spark.createDataFrame(
    [(1, "books", 1200), (2, "books", 800), (3, "games", 2500), (4, "games", 400)],
    "order_id INT, category STRING, amount_cents INT",
)
orders.show()


In [ ]:
totals = (
    orders.filter(F.col("amount_cents") >= 1000)
    .groupBy("category")
    .agg(F.sum("amount_cents").alias("total_cents"))
    .orderBy("category")
)
totals.show()
assert [(row.category, row.total_cents) for row in totals.collect()] == [
    ("books", 1200), ("games", 2500)
]


In [ ]:
orders.createOrReplaceTempView("orders")
spark.sql("""
    SELECT category, COUNT(*) AS order_count
    FROM orders GROUP BY category ORDER BY category
""").show()


In [ ]:
spark.stop()
